# Proyecto: Segmentación y Clasificación de Piezas LEGO con Machine Learning y Deep Learning
**Dataset:** LEGO Bricks (20 categorías de las familias Brick y Plate)  
**Modelos Evaluados:**
1. Perceptrón (One-vs-Rest)
2. SGAdaline (Stochastic Gradient Descent Adaline multiclase)
3. Support Vector Machine (SVM con RBF, Polinómico y Lineal)
4. Árboles de Decisión (CART con selección de hiperparámetros e importancia de características)
5. Redes Neuronales Artificiales (ANN / MLP en PyTorch con Dropout y BatchNorm)
6. Redes Neuronales Convolucionales (CNN en PyTorch con Data Augmentation)


In [ ]:
import sys
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Añadir raíz del proyecto al path
sys.path.insert(0, os.path.abspath('..'))
import config
from preprocessing.dataset_explorer import explore_dataset
from preprocessing.data_loader import create_or_load_splits
from segmentation.hsv_segmenter import HSVSegmenter
from features.feature_extractor import LEGOFeatureExtractor
from pipeline import LEGODetectionPipeline

print('PyTorch version:', torch.__version__)
print('OpenCV version:', cv2.__version__)
print('Dispositivo de cómputo:', config.DEVICE)


## 1. Exploración y Análisis del Dataset
Inspección de las 4,580 imágenes recortadas y 3,464 imágenes base, verificación de integridad, detección de duplicados por MD5 y correspondencia por marca de tiempo.

In [ ]:
summary, crop_recs, base_recs = explore_dataset()


## 2. Segmentación con OpenCV en el Espacio HSV
Segmentación combinando umbralización Otsu en el Canal S (piezas saturadas) y umbral inverso en Canal V (piezas oscuras), seguido de operaciones morfológicas y contornos.

In [ ]:
segmenter = HSVSegmenter()
sample_base_path = os.path.join(config.BASE_IMAGES_DIR, 'Brick_2x2_L', '2_Brick_2x2_L_180708213430.jpg')
img_bgr = cv2.imread(sample_base_path)

mask, pieces = segmenter.detect_and_extract_pieces(img_bgr, method='hsv_otsu')
overlay = segmenter.draw_detections(img_bgr, pieces, labels=['Brick_2x2_L'], confidences=[0.98])

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
ax[0].set_title('1. Base Original')
ax[0].axis('off')

ax[1].imshow(mask, cmap='gray')
ax[1].set_title('2. Máscara HSV Otsu')
ax[1].axis('off')

ax[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
ax[2].set_title('3. Detección y Contorno')
ax[2].axis('off')
plt.show()


## 3. Extracción de Características: Forma vs. Forma + Color
Carga de características precomputadas (301 dimensiones en Forma vs. 325 dimensiones en Forma + Color).

In [ ]:
feat_shape = np.load('../data/features_shape_only.npz')
feat_all = np.load('../data/features_shape_and_color.npz')

print('Dimensión características Forma:', feat_shape['X_train'].shape[1])
print('Dimensión características Forma + Color:', feat_all['X_train'].shape[1])


## 4. Tabla Comparativa de Rendimiento Experimental
Resultados cuantitativos sobre el conjunto de prueba (687 muestras) en las mismas condiciones experimentales.

In [ ]:
df_results = pd.read_csv('../results/models/models_comparison_table.csv')
display(df_results)


## 5. Visualizaciones Comparativas
Gráficos de rendimiento global, tiempos de entrenamiento y latencia de inferencia.

In [ ]:
img_bench = cv2.imread('../results/models/models_benchmark_comparison.png')
plt.figure(figsize=(16, 5))
plt.imshow(cv2.cvtColor(img_bench, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Comparación de Modelos: Accuracy, F1-Score, Train Time y Latencia', fontsize=13, fontweight='bold')
plt.show()

img_scen = cv2.imread('../results/models/shape_vs_color_comparison.png')
plt.figure(figsize=(10, 5))
plt.imshow(cv2.cvtColor(img_scen, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Impacto de Color vs. Solo Forma en Modelos Tradicionales', fontsize=13, fontweight='bold')
plt.show()


## 6. Pipeline End-to-End: Segmentación y Clasificación
Ejemplo de inferencia recibiendo una fotografía original, aislando la pieza y clasificándola mediante la red neuronal convolucional entrenada.

In [ ]:
pipeline = LEGODetectionPipeline(model_type='cnn')
sample_test_img = os.path.join(config.BASE_IMAGES_DIR, 'Plate_1x1', '2_Plate_1x1_180722193025.jpg')
res = pipeline.process_image(sample_test_img, segmentation_method='hsv_otsu', save_output=False)

plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(res['overlay'], cv2.COLOR_BGR2RGB))
plt.title(f"Detección: {res['pieces'][0]['class_name']} (Confianza: {res['pieces'][0]['confidence']*100:.1f}%)", fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()
